In [1]:
import numpy as np
import pandas as pd
import torch
print(torch.cuda.is_available())

True


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
df = pd.read_csv(
    "/content/updated_data.csv",
    engine="python",
    on_bad_lines="skip"
)

In [4]:
from sentence_transformers import SentenceTransformer

In [5]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device="cuda"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
response_embeddings = embedding_model.encode(
    df["response"].fillna("").tolist(),
    batch_size=64,
    show_progress_bar=True
)

Batches:   0%|          | 0/1494 [00:00<?, ?it/s]

In [6]:
def documents_to_text(documents):
    if isinstance(documents, (list, tuple, np.ndarray)):
        return " ".join(map(str, documents))
    return str(documents)

In [7]:
documents_text = df["documents"].apply(documents_to_text)

In [8]:
document_embeddings = embedding_model.encode(
    documents_text.tolist(),
    batch_size=64,
    show_progress_bar=True
)

Batches:   0%|          | 0/1494 [00:00<?, ?it/s]

In [10]:
question_embeddings = embedding_model.encode(
    df["question"].fillna("").tolist(),
    batch_size=64,
    show_progress_bar=True
)

Batches:   0%|          | 0/1494 [00:00<?, ?it/s]

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

doc_response_similarity = cosine_similarity(
    document_embeddings,
    response_embeddings
).diagonal()

question_response_similarity = cosine_similarity(
    question_embeddings,
    response_embeddings
).diagonal()

question_document_similarity = cosine_similarity(
    question_embeddings,
    document_embeddings
).diagonal()

In [ ]:
df["question_embeddings"]=list(question_embeddings)

df["document_embeddings"]=list(document_embeddings)

df["response_embeddings"]=list(response_embeddings)

df["doc_response_similarity"] = doc_response_similarity

df["question_response_similarity"] = question_response_similarity

df["question_document_similarity"] = question_document_similarity

In [ ]:
df.head()

In [ ]:
df.to_csv(r"C:\Users\BHUPATHI NADAR\OneDrive\Desktop\Main_project\MLSC-Task\Data\updated_semantic_data.csv", index=False)

In [11]:
df["response_embeddings"]=list(response_embeddings)

In [12]:
df.columns

Index(['question', 'documents', 'response', 'sentence_support_information',
       'label', 'response_embeddings'],
      dtype='object')

In [15]:
np.save("/content/drive/MyDrive/response_embeddings.npy", response_embeddings)

In [9]:
np.save("/content/drive/MyDrive/document_embeddings.npy", document_embeddings)

In [11]:
np.save("/content/drive/MyDrive/question_embeddings.npy", question_embeddings)